# Government Events Data Pipeline

The data is 601 Saudi public sector events (extracted from open data).


In [1]:
!pip install -q pyspark==3.5.3 delta-spark==3.2.0 kafka-python pydantic chromadb sentence-transformers rank-bm25 loguru great_expectations openlineage-python pandas pyarrow

## Ingestion (kafka + schema validation)

The producer sends all the events plus 2 bad messages on purpose, and the consumer
validates every message with pydantic. The bad ones get rejected.

In [2]:
import ingestion

ingestion.main()

2026-07-16 12:00:19.989 | INFO     | ingestion:load_data:40 - loaded 601 rows from the csv
2026-07-16 12:00:19.995 | INFO     | ingestion:main:159 - producing 603 messages to topic gov_events
2026-07-16 12:00:19.997 | WARNING  | ingestion:main:167 - kafka not available (no kafka broker on localhost:9092), using mock instead
2026-07-16 12:00:19.998 | DEBUG    | ingestion:validate_messages:108 - accepted message 0
2026-07-16 12:00:20.000 | DEBUG    | ingestion:validate_messages:108 - accepted message 1
2026-07-16 12:00:20.001 | DEBUG    | ingestion:validate_messages:108 - accepted message 2
2026-07-16 12:00:20.002 | DEBUG    | ingestion:validate_messages:108 - accepted message 3
2026-07-16 12:00:20.004 | DEBUG    | ingestion:validate_messages:108 - accepted message 4
2026-07-16 12:00:20.005 | DEBUG    | ingestion:validate_messages:108 - accepted message 5
2026-07-16 12:00:20.006 | DEBUG    | ingestion:validate_messages:108 - accepted message 6
2026-07-16 12:00:20.007 | DEBUG    | ingesti

---- Deliverable 1: ingestion ----


2026-07-16 12:00:20.182 | DEBUG    | ingestion:validate_messages:108 - accepted message 274
2026-07-16 12:00:20.183 | DEBUG    | ingestion:validate_messages:108 - accepted message 275
2026-07-16 12:00:20.184 | DEBUG    | ingestion:validate_messages:108 - accepted message 276
2026-07-16 12:00:20.185 | DEBUG    | ingestion:validate_messages:108 - accepted message 277
2026-07-16 12:00:20.185 | DEBUG    | ingestion:validate_messages:108 - accepted message 278
2026-07-16 12:00:20.186 | DEBUG    | ingestion:validate_messages:108 - accepted message 279
2026-07-16 12:00:20.187 | DEBUG    | ingestion:validate_messages:108 - accepted message 280
2026-07-16 12:00:20.187 | DEBUG    | ingestion:validate_messages:108 - accepted message 281
2026-07-16 12:00:20.188 | DEBUG    | ingestion:validate_messages:108 - accepted message 282
2026-07-16 12:00:20.188 | DEBUG    | ingestion:validate_messages:108 - accepted message 283
2026-07-16 12:00:20.189 | DEBUG    | ingestion:validate_messages:108 - accepted 

total messages: 603
accepted: 601
rejected: 2


## Delta Lakehouse (bronze / silver / gold)

Bronze is the raw data, silver is the cleaned data (real dates, no duplicates),
and gold is the summary tables. Also shows MERGE and schema enforcement.

In [3]:
import lakehouse

lakehouse.main()

---- Deliverable 2: lakehouse ----


2026-07-16 12:00:41.996 | SUCCESS  | lakehouse:main:74 - bronze saved with 601 rows
2026-07-16 12:00:42.292 | INFO     | lakehouse:main:87 - writing silver table...
2026-07-16 12:00:55.790 | SUCCESS  | lakehouse:main:89 - silver saved with 600 rows (duplicates removed)


+-------------+--------------------+------+----------+-------------+
|   request_id|              entity|  city|start_date|duration_days|
+-------------+--------------------+------+----------+-------------+
|GOV-2025-2193|المركز الوطني لتن...|الرياض|2026-01-04|            1|
|GOV-2025-2194| معهد الادارة العامة|الرياض|2026-01-04|            2|
|GOV-2025-2195| معهد الادارة العامة| الخبر|2026-01-01|            1|
|GOV-2026-0003|مكتب شؤون المهمات...|الرياض|2026-01-07|            1|
|GOV-2026-0005|المركز الوطني لتن...|الرياض|2026-01-05|            1|
+-------------+--------------------+------+----------+-------------+
only showing top 5 rows



2026-07-16 12:00:58.505 | INFO     | lakehouse:main:110 - running the merge...
2026-07-16 12:01:08.561 | SUCCESS  | lakehouse:main:117 - merge done: updated venue of GOV-2025-2193 and inserted GOV-2026-8888
2026-07-16 12:01:12.352 | INFO     | lakehouse:main:126 - building gold tables...


+-------------+---------------------+------+
|request_id   |venue                |city  |
+-------------+---------------------+------+
|GOV-2025-2193|فندق الفيصلية (تحديث)|الرياض|
|GOV-2026-8888|كراون بلازا          |الرياض|
+-------------+---------------------+------+

events by city:
+---------------+-----+
|           city|count|
+---------------+-----+
|         الرياض|  406|
|            جدة|   54|
|          الخبر|   48|
|          بريدة|   17|
|المدينة المنورة|   10|
|    مكة المكرمة|   10|
|         الطائف|    9|
|           ابها|    8|
|           الرس|    7|
|الجبيل الصناعية|    6|
+---------------+-----+
only showing top 10 rows

events by type:


2026-07-16 12:01:20.104 | SUCCESS  | lakehouse:main:137 - gold tables saved


+------------+-----+
|  event_type|count|
+------------+-----+
|دورة تدريبية|  234|
|    ورشة عمل|  196|
|      اجتماع|   82|
|  معرض توعوي|   47|
| معرض تعريفي|   14|
|       مؤتمر|   11|
|       منتدى|    8|
|        ندوة|    6|
|        مزاد|    2|
|      محاضرة|    1|
+------------+-----+

testing schema enforcement (this write should fail)...


2026-07-16 12:01:20.331 | SUCCESS  | lakehouse:main:152 - delta rejected the write because of the extra column, good


error was: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 6c755bab-a8fb-4b56-8ff2-a9e6e6231024).


## RAG pipeline

Every event becomes a document, then: chunking -> chromadb vector index -> bm25 ->
hybrid search (rrf) -> cross encoder reranking.

In [4]:
import rag_pipeline

rag_pipeline.main()

2026-07-16 12:01:33.593 | INFO     | rag_pipeline:make_documents:42 - made 600 documents
2026-07-16 12:01:33.598 | INFO     | rag_pipeline:build_vector_index:60 - building the vector index, first run downloads the model...


---- Deliverable 3: rag pipeline ----
600 documents -> 1805 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-16 12:02:31.786 | SUCCESS  | rag_pipeline:build_vector_index:72 - indexed 1805 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

2026-07-16 12:02:58.969 | SUCCESS  | rag_pipeline:main:167 - retrieval looks good


metrics: {'context_precision': 1.0, 'avg_similarity': 0.82}

query: ما تفاصيل الفعالية رقم GOV-2026-0040؟


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

2026-07-16 12:03:08.991 | SUCCESS  | rag_pipeline:main:167 - retrieval looks good
2026-07-16 12:03:08.992 | SUCCESS  | rag_pipeline:main:171 - rag pipeline done


top 3 results after reranking:
  1. المكان: مجمع غاليريا مول (مراكز تجارية) في مدينة الجبيل الصناعية. التاريخ من 4/29/2026 الى 4/30/2026
  2. المكان: متحف وبستان الصافية (مراكز تجارية) في مدينة المدينة المنورة. التاريخ من 5/16/2026 الى 5/16/2
  3. المكان: النخيل مول بمدينة الدمام (مراكز تجارية) في مدينة الدمام. التاريخ من 4/23/2026 الى 4/25/2026 
prompt for the llm (first 200 chars):
  اجب على السؤال من السياق التالي فقط.  السياق: [1] المكان: مجمع غاليريا مول (مراكز تجارية) في مدينة الجبيل الصناعية. التاريخ من 4/29/2026 الى 4/30/2026 ورقم الطلب GOV-2026-0782. [2] المكان: متحف وبستان
metrics: {'context_precision': 1.0, 'avg_similarity': 0.8}


## Quality gate (great expectations + openlineage)

Checks based on the 6 dimensions.

In [5]:
import quality_check

quality_check.main()

2026-07-16 12:03:09.012 | INFO     | quality_check:load_and_profile:41 - dropped 1 duplicate row(s)
2026-07-16 12:03:09.022 | SUCCESS  | quality_check:quality_checks:81 - completeness passed
2026-07-16 12:03:09.023 | SUCCESS  | quality_check:quality_checks:81 - accuracy passed
2026-07-16 12:03:09.024 | SUCCESS  | quality_check:quality_checks:81 - consistency passed
2026-07-16 12:03:09.024 | SUCCESS  | quality_check:quality_checks:81 - timeliness passed
2026-07-16 12:03:09.025 | SUCCESS  | quality_check:quality_checks:81 - uniqueness passed
2026-07-16 12:03:09.026 | SUCCESS  | quality_check:quality_checks:81 - validity passed


---- Deliverable 5: quality gate ----
rows: 601, columns: 9
nulls per column:
entity        0
event_type    0
title         0
start_date    0
end_date      0
venue_type    0
venue         0
city          0
request_id    0
duplicate rows: 1
duplicate request ids: 1

quality report:
  [PASS] completeness: 0 nulls in important columns
  [PASS] accuracy: 0 events with weird duration
  [PASS] consistency: 0 request ids with more than one city
  [PASS] timeliness: 0 events outside 2026-2027
  [PASS] uniqueness: 0 duplicated request ids
  [PASS] validity: 0 request ids with wrong format


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpweyr4b8q' for ephemeral docs site


Calculating Metrics:   0%|          | 0/23 [00:00<?, ?it/s]

2026-07-16 12:03:11.923 | SUCCESS  | quality_check:main:154 - quality gate PASSED, 600 rows saved to production/
2026-07-16 12:03:12.027 | SUCCESS  | quality_check:emit_lineage:140 - openlineage events saved to lineage/events.log (600 rows)



great expectations result: success=True
  [PASS] expect_column_values_to_not_be_null
  [PASS] expect_column_values_to_match_regex
  [PASS] expect_column_values_to_not_be_null
  [PASS] expect_column_values_to_not_be_null


True

## Orchestration (airflow dag)

The dag connects everything in order: ingest -> lakehouse -> quality gate -> rag.

In [6]:
import pipeline_dag

print("airflow installed:", pipeline_dag.HAS_AIRFLOW)
for name, fn in pipeline_dag.TASKS:
    print("task:", name)

airflow installed: False
task: ingest
task: lakehouse
task: quality_gate
task: rag


Now run the full pipeline from start to end (this runs the 4 modules again):

In [7]:
pipeline_dag.main()

2026-07-16 12:03:12.049 | WARNING  | pipeline_dag:main:73 - airflow not installed, running the tasks in order manually
2026-07-16 12:03:12.050 | INFO     | pipeline_dag:run_without_airflow:62 - starting task ingest
2026-07-16 12:03:12.058 | INFO     | ingestion:load_data:40 - loaded 601 rows from the csv
2026-07-16 12:03:12.062 | INFO     | ingestion:main:159 - producing 603 messages to topic gov_events
2026-07-16 12:03:12.064 | WARNING  | ingestion:main:167 - kafka not available (no kafka broker on localhost:9092), using mock instead
2026-07-16 12:03:12.065 | DEBUG    | ingestion:validate_messages:108 - accepted message 0
2026-07-16 12:03:12.066 | DEBUG    | ingestion:validate_messages:108 - accepted message 1
2026-07-16 12:03:12.067 | DEBUG    | ingestion:validate_messages:108 - accepted message 2
2026-07-16 12:03:12.068 | DEBUG    | ingestion:validate_messages:108 - accepted message 3
2026-07-16 12:03:12.070 | DEBUG    | ingestion:validate_messages:108 - accepted message 4
2026-07-1

---- Deliverable 4: orchestration ----

########## task: ingest ##########
---- Deliverable 1: ingestion ----


2026-07-16 12:03:12.118 | DEBUG    | ingestion:validate_messages:108 - accepted message 61
2026-07-16 12:03:12.119 | DEBUG    | ingestion:validate_messages:108 - accepted message 62
2026-07-16 12:03:12.119 | DEBUG    | ingestion:validate_messages:108 - accepted message 63
2026-07-16 12:03:12.120 | DEBUG    | ingestion:validate_messages:108 - accepted message 64
2026-07-16 12:03:12.121 | DEBUG    | ingestion:validate_messages:108 - accepted message 65
2026-07-16 12:03:12.122 | DEBUG    | ingestion:validate_messages:108 - accepted message 66
2026-07-16 12:03:12.122 | DEBUG    | ingestion:validate_messages:108 - accepted message 67
2026-07-16 12:03:12.123 | DEBUG    | ingestion:validate_messages:108 - accepted message 68
2026-07-16 12:03:12.124 | DEBUG    | ingestion:validate_messages:108 - accepted message 69
2026-07-16 12:03:12.125 | DEBUG    | ingestion:validate_messages:108 - accepted message 70
2026-07-16 12:03:12.126 | DEBUG    | ingestion:validate_messages:108 - accepted message 71

total messages: 603
accepted: 601
rejected: 2

########## task: lakehouse ##########
---- Deliverable 2: lakehouse ----


2026-07-16 12:03:14.681 | SUCCESS  | lakehouse:main:74 - bronze saved with 601 rows
2026-07-16 12:03:14.749 | INFO     | lakehouse:main:87 - writing silver table...
2026-07-16 12:03:21.169 | SUCCESS  | lakehouse:main:89 - silver saved with 600 rows (duplicates removed)


...

2026-07-16 12:04:58.686 | SUCCESS  | rag_pipeline:main:167 - retrieval looks good


top 3 results after reranking:
  1. نظمت مدينة الملك سعود الطيبة فعالية اجتماع بعنوان إجتماع اللقاء السنوي. المكان: منتجع ماربيلا (مرافق
  2. نظمت معهد الادارة العامة فعالية ورشة عمل بعنوان كتابة الكراسات الحكومية. المكان: فندق هيلتون جارد ان
  3. نظمت الهيئة العامة للنقل فعالية اجتماع بعنوان اجتماعات الوفد الاردني. المكان: فندق ماريوت الرياض (فن
prompt for the llm (first 200 chars):
  اجب على السؤال من السياق التالي فقط.  السياق: [1] نظمت مدينة الملك سعود الطيبة فعالية اجتماع بعنوان إجتماع اللقاء السنوي. المكان: منتجع ماربيلا (مرافق عامة) في مدينة الرياض. [2] نظمت معهد الادارة العا
metrics: {'context_precision': 1.0, 'avg_similarity': 0.82}

query: ما تفاصيل الفعالية رقم GOV-2026-0040؟


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

2026-07-16 12:05:07.048 | SUCCESS  | rag_pipeline:main:167 - retrieval looks good
2026-07-16 12:05:07.049 | SUCCESS  | rag_pipeline:main:171 - rag pipeline done
2026-07-16 12:05:07.053 | SUCCESS  | pipeline_dag:run_without_airflow:64 - task rag done


top 3 results after reranking:
  1. المكان: مجمع غاليريا مول (مراكز تجارية) في مدينة الجبيل الصناعية. التاريخ من 4/29/2026 الى 4/30/2026
  2. المكان: متحف وبستان الصافية (مراكز تجارية) في مدينة المدينة المنورة. التاريخ من 5/16/2026 الى 5/16/2
  3. المكان: النخيل مول بمدينة الدمام (مراكز تجارية) في مدينة الدمام. التاريخ من 4/23/2026 الى 4/25/2026 
prompt for the llm (first 200 chars):
  اجب على السؤال من السياق التالي فقط.  السياق: [1] المكان: مجمع غاليريا مول (مراكز تجارية) في مدينة الجبيل الصناعية. التاريخ من 4/29/2026 الى 4/30/2026 ورقم الطلب GOV-2026-0782. [2] المكان: متحف وبستان
metrics: {'context_precision': 1.0, 'avg_similarity': 0.8}
